In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
from sklearn.metrics import confusion_matrix, f1_score, classification_report, precision_score, recall_score
from typing import List

%run classes_axillary.ipynb
#import package.classes_axillary

%run evaluationMetrics.ipynb
#from evaluationMetrics import getTP_FP

%run utilities.ipynb
#import utilities as util
#from utilities import create_eval_parameters, create_configuration_set
#from package.detection_change_points import change_point_detection, change_point_detection_load


%run configurations.ipynb
#import configurations as config





def extract_unique_sorted_integers(df, column_name):
    integer_list = []

    # Iterate over each element in the specified column
    for row_data in df[column_name]:
        row_integers = []
        for item in row_data:
            integers = item.strip('[]').split(',')
            for integer in integers:
                row_integers.append(int(integer))

        row_integers_unique = list(set(row_integers))
        row_integers_unique_sorted = sorted(row_integers_unique)
        integer_list.append(row_integers_unique_sorted)

    df['change_point'] = integer_list
    del df[column_name]

    return df


def get_detected_change_point_and_type(results, eval_parameters):
    log_name = eval_parameters['log_name']
    noise_level = eval_parameters['noise_level']

    # complexity = eval_parameters['complexity']
    # condition = (results.log_name == log_name) & (results.eval_complexity == complexity) & (
    #    results.eval_noise_level == noise_level)
    condition = (results.log_name == log_name) & (results.eval_noise_level == noise_level)
    change_points_log_name = results[condition].calc_change_index.tolist()
    if 'na' in change_points_log_name or len(change_points_log_name) == 0:
        change_points_log_name = []
        change_types_log_name = []
        change_drift_log_name = []
    else:
        change_points_log_name = [int(x) for x in results[condition].calc_change_index.tolist()]
        #change_points_log_name = [x if type(x) == int else round(eval(x)) for x in
        #                          results[condition].calc_change_index.tolist()]
        change_types_log_name = results[condition].calc_change_type.tolist()
        change_drift_log_name = results[condition].calc_drift_type.tolist()

    return list(zip(change_points_log_name, change_types_log_name, change_drift_log_name))


def get_actual_change_point_and_type(gold_standard, eval_parameters):
    log_name = eval_parameters['log_name']
    change_points_log_name = gold_standard[gold_standard.log_name == log_name].change_point.values[0]
    change_types_log_name = gold_standard[gold_standard.log_name == log_name].change_type.values[0]
    change_drift_log_name = gold_standard[gold_standard.log_name == log_name].drift_type.values[0]

    return list(zip(change_points_log_name, change_types_log_name, change_drift_log_name))


def calc_log_size(results, log_name):
    condition = (results.log_name == log_name)
    log_size = results[condition].log_size.values[0]
    return log_size


def update_global_accuracy_dict(accuracy_dict, eval_focus, TP, FP, FN_TP):
    accuracy_dict[eval_focus]['TP'] += TP
    accuracy_dict[eval_focus]['FP'] += FP
    accuracy_dict[eval_focus]['FN_TP'] += FN_TP
    return accuracy_dict


def create_global_accuracy_dict(change_types, measure):
    accuracy_dict = {}

    for key in change_types:
        accuracy_dict[key] = {}
        for subkey in measure:
            accuracy_dict[key][subkey] = 0

    return accuracy_dict


def get_precision(TP, FP):
    precision = np.where(TP + FP > 0, np.divide(TP, TP + FP), 0)
    return precision


def get_recall(TP, FN_TP):
    recall = np.where(FN_TP > 0, np.divide(TP, FN_TP), 0)
    return recall


def get_f1_score(precision, recall):
    try:
        f1_score = (2 * precision * recall) / (precision + recall)
        return f1_score
    except:
        print("Calculation of F1-Score resulted in division by 0.")
        f1_score = np.NaN

    return f1_score


def _weighted_f1_score(group):
    fn_tp_weighted_f1 = (group['F1_score'] * group['FN_TP']).sum() / group['FN_TP'].sum()
    return pd.Series({'Weighted_F1': fn_tp_weighted_f1})


def _get_weighted_F1_accuracy(report):
    # Group by 'noise_level', 'lag', and any other columns that are constant for each group
    grouped = report.groupby(['noise_level', 'lag'])

    # Apply the custom function to calculate the weighted F1 score for each group
    temp = grouped.apply(_weighted_f1_score).reset_index()

    weighted_F1_per_set = temp[temp.noise_level != 'with_noise_15']

    return weighted_F1_per_set


def main_hyperparameter(step):
    results = pd.read_csv(Path(dir_input, results_file_name))

    best_group_name = None
    best_weighted_F1_avg = 0.0
    best_report = pd.DataFrame()
    best_weighted_F1_per_set = pd.DataFrame()

    results_all = []

    grouping_condition = ['config_constant_alpha', 'config_noise', 'config_incremental', 'config_recurring', ]
    results_grouped_df = results.groupby(grouping_condition)

    # Iterate over the groups and perform operations
    i = 0
    for group_name, group_data in results_grouped_df:
        i += 1
        print(f"{i}/{len(results_grouped_df)} group: {group_name}", flush=True)
        report = main(step, evaluation_lags, results=group_data, save=False)

        weighted_F1_per_set = _get_weighted_F1_accuracy(report)
        weighted_F1_per_set_avg = np.mean(weighted_F1_per_set.Weighted_F1.values)

        if weighted_F1_per_set_avg > best_weighted_F1_avg:
            print(f"Better parameters found: {group_name}")
            print(weighted_F1_per_set)
            print(weighted_F1_per_set_avg)
            print()
            best_group_name = group_name
            best_weighted_F1_avg = weighted_F1_per_set_avg
            best_report = report
            best_weighted_F1_per_set = weighted_F1_per_set

        results_all.append({'group': group_name,
                            'accuracy': weighted_F1_per_set_avg,
                            'results_weighted': weighted_F1_per_set,
                            'results': report})
        timing.report_interim()

    file_name = Path(dir_output,
                     f'{results_file_name[:-4]}_best_report_step_{step}_{best_group_name}_{best_weighted_F1_avg}.csv')
    best_report.to_csv(file_name)
    file_name_agg = Path(dir_output,
                         f'{results_file_name[:-4]}_best_report_step_{step}_{best_group_name}_{best_weighted_F1_avg}_agg.csv')
    best_weighted_F1_per_set.to_csv(file_name_agg)
    file_name_pck = Path(dir_output, f'{results_file_name[:-4]}_interim_results_{step}.pkl')
    with open(file_name_pck, 'wb') as file:
        pickle.dump(results_all, file)

    print(best_group_name, best_weighted_F1_avg)
    print(best_report)
    print(best_weighted_F1_per_set)

    return best_group_name, best_weighted_F1_avg


def get_total_evaluation_results(evaluation_report):
    aggregations = {
        'TP': 'sum',
        'FP': 'sum',
        'FN_TP': 'sum'}
    # grouping = ['noise_level', 'complexity', 'method', 'window_size', 'lag', 'eval_focus']
    #grouping = ['noise_level', 'method', 'window_size', 'lag', 'eval_focus']
    grouping = ['noise_level', 'method', 'lag', 'eval_focus']
    evaluation_report_agg = evaluation_report.groupby(grouping).agg(aggregations)

    evaluation_report_agg = evaluation_report_agg.assign(Precision=lambda x: get_precision(x['TP'], x['FP']))
    evaluation_report_agg = evaluation_report_agg.assign(Recall=lambda x: get_recall(x['TP'], x['FN_TP']))
    evaluation_report_agg = evaluation_report_agg.assign(F1_score=lambda x: get_f1_score(x['Precision'], x['Recall']))

    return evaluation_report_agg

C:\Users\la1949\AppData\Local\anaconda3\envs\ProcessMining\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pm4py.objects.log.importer.xes import importer as xes_importer


def evalFunctionGraph(log_names, system_paths, driftAlgo, lag):

    precisionList = []
    recallList = []
    f1List = []
    
    for path in system_paths:
        for name in log_names:
            file_path = os.path.join(path, name)
        
            if not os.path.isfile(file_path):
                #print(f" File not found: {file_path}")
                continue

            print(f" File found: {file_path}")
            #difference to evalFunction:
            #logXES = load_event_log(file_path)  # OLD
            logVar = pm4py.read.read_xes(file_path)
            logXES = logVar.groupby(['case:concept:name'])['concept:name'].apply(list).reset_index()
            logXES["case:concept:name"] = logXES["case:concept:name"].apply(lambda x: int(x))
            logXES = logXES.sort_values(by=["case:concept:name"])
            #print(logXES)
            

            #get detcted and actual change points
            detectedCP =  driftAlgo(logXES) # apply concept drift detection algorithm
            print('detected CPs:', detectedCP)
            #TEST
            #detectedCP = [np.int64(1505), np.int64(2215), np.int64(2840)]
            
            actualCP = goldStandard.loc[goldStandard['log_name'] == name,'change_point'].explode().tolist()
            print('actual CPs:', actualCP)
            
            #According to Kraus lag is calculated based on the number of cases in event log (log_size)
            log_size = len(logXES)
            print('logSize', log_size)
            lag_acc = int(log_size / 100 * lag)

            #returns Tuple[Union[float,np.NaN], Union[float,np.NaN]]: A tuple (precision, recall)
            precision, recall = calcPrecisionRecall(detectedCP, actualCP, lag_acc, zero_division=0)
            f1 = F1_Score(detectedCP, actualCP, lag_acc, zero_division=0)
            

            precisionList.append(precision)
            print(precisionList)
            recallList.append(recall)
            print(recallList)
            f1List.append(f1)
            print(f1List)
    
    print('------------------')
    print('Evaluation Measures:', driftAlgo)
    print('------------------')
    print('precision:', np.average(precisionList), '-> full list:', precisionList)
    print('recall:', np.average(recallList), '-> full list:', recallList)
    print('f1:', np.average(f1List), '-> full list', f1List)
    print('------------------')
    #return 'precision:', precisionList, np.average(precisionList), 'recall:', recallList, np.average(recallList), 'f1:', f1List, np.average(f1List)
            



In [3]:
import ast
goldStandard = pd.read_csv('data_collection\datasets_evaluation\gold_standard.csv')
goldStandard["change_point"] = goldStandard["change_point"].apply(ast.literal_eval)
#goldStandard

In [4]:
#2 approach
import ruptures as rpt
from sklearn.preprocessing import RobustScaler

def generate_timeS(eventLog, distance_function, window_size):

    #eventLog = eventLog.sort_values(by=["case:concept:name"])
    
    w_1 = {}
    w_2 = {}
    for i in range(len(eventLog) - 2*window_size+1):
        w_1[i+window_size] = eventLog['concept:name'][i:i+window_size].reset_index(drop=True)
        w_2[i+window_size] = eventLog['concept:name'][i+window_size:i+2*window_size].reset_index(drop=True)
    
    distances = []
    index = []
    
    for j in range(len(eventLog)):
        if j in w_1:
            distance = distance_function(w_1[j], w_2[j])
        else:
            distance = np.nan    
        distances.append(distance)
        index.append(j) #erstes Element aus zweiter Stichprobe als Change Point festgelegt   
    
    return pd.Series(distances, index=index)

#test
#eventLog = logVar
#generate_timeS(eventLog, cosineSim1_multi,10)


# z-score standardization
def zStandard(vec):
    if vec.std() != 0:
        vec_st = (vec - vec.mean()) / vec.std()
        return vec_st
    else:
        return vec
        



def multiTimeSeries(event_log):
    window = 200
    
    x_win1 = generate_timeS(event_log, cosineSim1_multi, window)
    x_win2 = generate_timeS(event_log, cosineSim2_multi, window)
    x_win3 = generate_timeS(event_log, cosineGraphSim_multi, window)

    xNan = np.array(x_win1)
    yNan = np.array(x_win2)
    zNan = np.array(x_win3)

    x = xNan[~np.isnan(xNan)]
    y = yNan[~np.isnan(yNan)]
    z = zNan[~np.isnan(zNan)]

    #x_st = zStandard(x)
    #y_st = zStandard(y)
    #z_st = zStandard(z)

    #multiArray = np.vstack([x_st,y_st,z_st]).T  # shape (n, p)
    multiArray = np.vstack([x,y,z]).T  # shape (n, p)

    return multiArray



# Full Evaluation

In [8]:
#According to Kraus lag is calculated based on the number of cases in an event log (log_size)
#According to Adamds lag_acc = 200 for all event logs
%run ConceptDriftBPM.ipynb

lags = 5 #[1, 5, 10]

base = Path("data_collection\datasets_evaluation\without_noise")
systemPaths = [
    os.path.join(base, "without_noise_part_1"),
    os.path.join(base, "without_noise_part_2") ]

log_names = goldStandard.log_name.tolist()
testLogs = log_names

def apply_kernelCPD(event_log):
    pen = len(event_log)**0.5
    multiTS = multiTimeSeries(event_log) 
    algo = rpt.KernelCPD(kernel="rbf").fit(multiTS)
    change_points = algo.predict(pen=pen) # Penalty controls number of changepoints (higher penalty = fewer changes)
    del change_points[-1]    
    return change_points

evalFunctionGraph(testLogs, systemPaths, apply_kernelCPD, lags)

 File found: data_collection\datasets_evaluation\without_noise\without_noise_part_1\log_10_1692952190.xes


parsing log, completed traces :: 100%|██████████| 7405/7405 [00:02<00:00, 2654.07it/s]


detected CPs: [np.int32(869), np.int32(1372), np.int32(2475), np.int32(2788), np.int32(3974), np.int32(4253), np.int32(5177), np.int32(6069)]
actual CPs: [1029, 1562, 2818, 4306, 5516, 6341]
logSize 7405
[0.75]
[1.0]
[0.8571428571428571]
 File found: data_collection\datasets_evaluation\without_noise\without_noise_part_1\log_11_1692952195.xes


parsing log, completed traces :: 100%|██████████| 3059/3059 [00:00<00:00, 8842.74it/s]


detected CPs: [np.int32(620), np.int32(1016), np.int32(1579), np.int32(1762)]
actual CPs: [1228, 1853]
logSize 3059
[0.75, 0.25]
[1.0, 0.5]
[0.8571428571428571, 0.3333333333333333]
 File found: data_collection\datasets_evaluation\without_noise\without_noise_part_1\log_12_1692952196.xes


...

detected CPs: [np.int32(1000), np.int32(1314), np.int32(2021), np.int32(2348), np.int32(3450), np.int32(4182), np.int32(5035), np.int32(5340), np.int32(6215), np.int32(6956)]
actual CPs: [1365, 2387, 3660, 4381, 5395, 6560, 7200]
logSize 8199
[0.75, 0.25, 0.6666666666666666, 0.6666666666666666, 0.8, 0.75, 0.6666666666666666, 0.8571428571428571, 0.7142857142857143, 0.8, 0.25, 0.6666666666666666, 1.0, 0.6666666666666666, 0.9166666666666666, 0.5714285714285714, 0.75, 0.8461538461538461, 0.625, 0.5, 1.0, 0.25, 0.5, 0.5, 0, 0.75, 0.5, 1.0, 0.8181818181818182, 0.3333333333333333, 0.9, 0.6666666666666666, 0.0, 0.5714285714285714, 0.5555555555555556, 0.8, 0.8333333333333334, 0.7142857142857143, 0.25, 0, 0.0, 0.42857142857142855, 0.9166666666666666, 0.5, 0.5833333333333334, 0.5, 0.6923076923076923, 0.7692307692307693, 0.375, 1.0, 0.2, 0.8333333333333334, 0.375, 0.0, 0.6363636363636364, 0.6666666666666666, 0.6, 0.7142857142857143, 0.5, 0.5, 0.75, 0.7333333333333333, 1.0, 0.5714285714285714, 0.66

parsing log, completed traces :: 100%|██████████| 11091/11091 [00:02<00:00, 4490.23it/s]


detected CPs: [np.int32(951), np.int32(1959), np.int32(3165), np.int32(3512), np.int32(4397), np.int32(4654), np.int32(5191), np.int32(6138), np.int32(7208), np.int32(8359), np.int32(8703), np.int32(9350)]
actual CPs: [1161, 2070, 3547, 4627, 5403, 6514, 7405, 8737, 9638]
logSize 11091
[0.75, 0.25, 0.6666666666666666, 0.6666666666666666, 0.8, 0.75, 0.6666666666666666, 0.8571428571428571, 0.7142857142857143, 0.8, 0.25, 0.6666666666666666, 1.0, 0.6666666666666666, 0.9166666666666666, 0.5714285714285714, 0.75, 0.8461538461538461, 0.625, 0.5, 1.0, 0.25, 0.5, 0.5, 0, 0.75, 0.5, 1.0, 0.8181818181818182, 0.3333333333333333, 0.9, 0.6666666666666666, 0.0, 0.5714285714285714, 0.5555555555555556, 0.8, 0.8333333333333334, 0.7142857142857143, 0.25, 0, 0.0, 0.42857142857142855, 0.9166666666666666, 0.5, 0.5833333333333334, 0.5, 0.6923076923076923, 0.7692307692307693, 0.375, 1.0, 0.2, 0.8333333333333334, 0.375, 0.0, 0.6363636363636364, 0.6666666666666666, 0.6, 0.7142857142857143, 0.5, 0.5, 0.75, 0.733

parsing log, completed traces :: 100%|██████████| 8290/8290 [00:01<00:00, 4524.39it/s]


detected CPs: [np.int32(685), np.int32(1067), np.int32(1851), np.int32(2232), np.int32(2866), np.int32(3902), np.int32(4728), np.int32(6138), np.int32(6915)]
actual CPs: [1091, 2218, 3016, 4029, 4956, 6397, 7026]
logSize 8290
[0.75, 0.25, 0.6666666666666666, 0.6666666666666666, 0.8, 0.75, 0.6666666666666666, 0.8571428571428571, 0.7142857142857143, 0.8, 0.25, 0.6666666666666666, 1.0, 0.6666666666666666, 0.9166666666666666, 0.5714285714285714, 0.75, 0.8461538461538461, 0.625, 0.5, 1.0, 0.25, 0.5, 0.5, 0, 0.75, 0.5, 1.0, 0.8181818181818182, 0.3333333333333333, 0.9, 0.6666666666666666, 0.0, 0.5714285714285714, 0.5555555555555556, 0.8, 0.8333333333333334, 0.7142857142857143, 0.25, 0, 0.0, 0.42857142857142855, 0.9166666666666666, 0.5, 0.5833333333333334, 0.5, 0.6923076923076923, 0.7692307692307693, 0.375, 1.0, 0.2, 0.8333333333333334, 0.375, 0.0, 0.6363636363636364, 0.6666666666666666, 0.6, 0.7142857142857143, 0.5, 0.5, 0.75, 0.7333333333333333, 1.0, 0.5714285714285714, 0.6666666666666666, 0

In [54]:
#According to Kraus lag is calculated based on the number of cases in an event log (log_size)
#According to Adamds lag_acc = 200 for all event logs
%run ConceptDriftBPM.ipynb

lags = 5 #[1, 5, 10]

base = Path("data_collection\datasets_evaluation\with_noise_5")
systemPaths = [
    os.path.join(base, "with_noise_5_part_1"),
    os.path.join(base, "with_noise_5_part_2") ]

testLogs = log_names

def apply_kernelCPD(event_log):
    pen = len(event_log)**0.5
    multiTS = multiTimeSeries(event_log) 
    algo = rpt.KernelCPD(kernel="rbf").fit(multiTS)
    change_points = algo.predict(pen=pen) # Penalty controls number of changepoints (higher penalty = fewer changes)
    return change_points

evalFunctionGraph(testLogs, systemPaths, apply_kernelCPD, lags)

 File found: data_collection\datasets_evaluation\with_noise_5\with_noise_5_part_1\log_10_1692952190.xes


parsing log, completed traces :: 100%|██████████| 7405/7405 [00:02<00:00, 2561.28it/s]


detected CPs: [np.int32(775), np.int32(1243), np.int32(2258), np.int32(2557), np.int32(3651), np.int32(3921), np.int32(4759), np.int32(5587), 6443]
actual CPs: [1029, 1562, 2818, 4306, 5516, 6341]
logSize 6842
[0.5555555555555556]
[0.8333333333333334]
[0.6666666666666667]
 File found: data_collection\datasets_evaluation\with_noise_5\with_noise_5_part_1\log_11_1692952195.xes


parsing log, completed traces :: 100%|██████████| 3059/3059 [00:00<00:00, 7601.46it/s]


detected CPs: [np.int32(446), np.int32(929), np.int32(1445), np.int32(1570), 2445]
actual CPs: [1228, 1853]
logSize 2844
[0.5555555555555556, 0.0]
[0.8333333333333334, 0.0]
[0.6666666666666667, 0]
 File found: data_collection\datasets_evaluation\with_noise_5\with_noise_5_part_1\log_12_1692952196.xes


...

detected CPs: [np.int32(888), np.int32(1197), np.int32(1757), np.int32(2159), np.int32(3157), np.int32(3805), np.int32(4614), np.int32(4907), np.int32(5681), np.int32(6393), 7124]
actual CPs: [1365, 2387, 3660, 4381, 5395, 6560, 7200]
logSize 7523
[0.5555555555555556, 0.0, 0.6666666666666666, 0.7142857142857143, 0.5714285714285714, 0.4444444444444444, 0.8333333333333334, 0.4444444444444444, 0.5, 0.42857142857142855, 0.2, 1.0, 0.42857142857142855, 0.7272727272727273, 0.8, 0.25, 0.6666666666666666, 0.8461538461538461, 0.2857142857142857, 0.2, 1.0, 0.2, 0.0, 0.3333333333333333, 1.0, 0.5555555555555556, 0.2, 0.8888888888888888, 0.9, 0.16666666666666666, 0.7272727272727273, 0.14285714285714285, 0.0, 0.18181818181818182, 0.3333333333333333, 0.8461538461538461, 0.5, 0.625, 0.0, 0.2857142857142857, 0.0, 0.14285714285714285, 0.8461538461538461, 0.3333333333333333, 0.38461538461538464, 0.3333333333333333, 0.6923076923076923, 0.8, 0.2857142857142857, 0.75, 0.2, 0.375, 0.14285714285714285, 0.16666

parsing log, completed traces :: 100%|██████████| 11091/11091 [00:02<00:00, 4284.62it/s]


detected CPs: [np.int32(877), np.int32(1782), np.int32(2901), np.int32(3233), np.int32(4022), np.int32(4303), np.int32(4808), np.int32(5653), np.int32(5996), np.int32(6620), np.int32(7705), np.int32(8038), np.int32(8598), 9831]
actual CPs: [1161, 2070, 3547, 4627, 5403, 6514, 7405, 8737, 9638]
logSize 10230
[0.5555555555555556, 0.0, 0.6666666666666666, 0.7142857142857143, 0.5714285714285714, 0.4444444444444444, 0.8333333333333334, 0.4444444444444444, 0.5, 0.42857142857142855, 0.2, 1.0, 0.42857142857142855, 0.7272727272727273, 0.8, 0.25, 0.6666666666666666, 0.8461538461538461, 0.2857142857142857, 0.2, 1.0, 0.2, 0.0, 0.3333333333333333, 1.0, 0.5555555555555556, 0.2, 0.8888888888888888, 0.9, 0.16666666666666666, 0.7272727272727273, 0.14285714285714285, 0.0, 0.18181818181818182, 0.3333333333333333, 0.8461538461538461, 0.5, 0.625, 0.0, 0.2857142857142857, 0.0, 0.14285714285714285, 0.8461538461538461, 0.3333333333333333, 0.38461538461538464, 0.3333333333333333, 0.6923076923076923, 0.8, 0.285

parsing log, completed traces :: 100%|██████████| 8290/8290 [00:01<00:00, 5807.14it/s]


detected CPs: [np.int32(588), np.int32(971), np.int32(1677), np.int32(2042), np.int32(2651), np.int32(3657), np.int32(4310), np.int32(5621), np.int32(6331), 7208]
actual CPs: [1091, 2218, 3016, 4029, 4956, 6397, 7026]
logSize 7607
[0.5555555555555556, 0.0, 0.6666666666666666, 0.7142857142857143, 0.5714285714285714, 0.4444444444444444, 0.8333333333333334, 0.4444444444444444, 0.5, 0.42857142857142855, 0.2, 1.0, 0.42857142857142855, 0.7272727272727273, 0.8, 0.25, 0.6666666666666666, 0.8461538461538461, 0.2857142857142857, 0.2, 1.0, 0.2, 0.0, 0.3333333333333333, 1.0, 0.5555555555555556, 0.2, 0.8888888888888888, 0.9, 0.16666666666666666, 0.7272727272727273, 0.14285714285714285, 0.0, 0.18181818181818182, 0.3333333333333333, 0.8461538461538461, 0.5, 0.625, 0.0, 0.2857142857142857, 0.0, 0.14285714285714285, 0.8461538461538461, 0.3333333333333333, 0.38461538461538464, 0.3333333333333333, 0.6923076923076923, 0.8, 0.2857142857142857, 0.75, 0.2, 0.375, 0.14285714285714285, 0.16666666666666666, 0.5

In [7]:
#According to Kraus lag is calculated based on the number of cases in an event log (log_size)
#According to Adamds lag_acc = 200 for all event logs
%run ConceptDriftBPM.ipynb

lags = 5 #[1, 5, 10]

base = Path("data_collection\datasets_evaluation\with_noise_10")
systemPaths = [
    os.path.join(base, "with_noise_10_part_1"),
    os.path.join(base, "with_noise_10_part_2") ]

log_names = goldStandard.log_name.tolist()
testLogs = log_names

def apply_kernelCPD(event_log):
    pen = len(event_log)**0.5
    multiTS = multiTimeSeries(event_log) 
    algo = rpt.KernelCPD(kernel="rbf").fit(multiTS)
    change_points = algo.predict(pen=pen) # Penalty controls number of changepoints (higher penalty = fewer changes)
    del change_points[-1]    
    return change_points

evalFunctionGraph(testLogs, systemPaths, apply_kernelCPD, lags)

 File found: data_collection\datasets_evaluation\with_noise_10\with_noise_10_part_1\log_10_1692952190.xes


parsing log, completed traces :: 100%|██████████| 7405/7405 [00:03<00:00, 2340.57it/s]


detected CPs: [np.int32(722), np.int32(1100), np.int32(2058), np.int32(2341), np.int32(3328), np.int32(3642), np.int32(4264), np.int32(5159)]
actual CPs: [1029, 1562, 2818, 4306, 5516, 6341]
logSize 6295
[0.25]
[0.3333333333333333]
[0.28571428571428575]
 File found: data_collection\datasets_evaluation\with_noise_10\with_noise_10_part_1\log_11_1692952195.xes


parsing log, completed traces :: 100%|██████████| 3059/3059 [00:00<00:00, 9285.03it/s]


detected CPs: [np.int32(381), np.int32(832), np.int32(1308), np.int32(1463)]
actual CPs: [1228, 1853]
logSize 2604
[0.25, 0.25]
[0.3333333333333333, 0.5]
[0.28571428571428575, 0.3333333333333333]
 File found: data_collection\datasets_evaluation\with_noise_10\with_noise_10_part_1\log_12_1692952196.xes


...

detected CPs: [np.int32(753), np.int32(1083), np.int32(1604), np.int32(1980), np.int32(2905), np.int32(3567), np.int32(4239), np.int32(4557), np.int32(5261), np.int32(5901)]
actual CPs: [1365, 2387, 3660, 4381, 5395, 6560, 7200]
logSize 6968
[0.25, 0.25, 0.5, 0.5, 0.5833333333333334, 0.625, 0.6, 0.5, 0.8571428571428571, 0.6, 0.25, 0.75, 0.3333333333333333, 0.5, 0.6428571428571429, 0.2857142857142857, 0.7142857142857143, 0.6923076923076923, 0.3333333333333333, 0.3333333333333333, 0.6363636363636364, 0.14285714285714285, 0.0, 0.4444444444444444, 1.0, 0.3333333333333333, 0.2, 0.7777777777777778, 0.6111111111111112, 0.0, 0.6, 0.2857142857142857, 0.0, 0.42857142857142855, 0.14285714285714285, 0.7142857142857143, 0.3333333333333333, 0.5, 0.14285714285714285, 0.0, 0.0, 0.16666666666666666, 0.6923076923076923, 0.5, 0.5, 0.42857142857142855, 0.5833333333333334, 0.7272727272727273, 0.375, 0.3333333333333333, 0.14285714285714285, 0.5, 0.2857142857142857, 0.5, 0.45454545454545453, 0.2, 0.466666666

parsing log, completed traces :: 100%|██████████| 11091/11091 [00:02<00:00, 3949.89it/s]


detected CPs: [np.int32(782), np.int32(1662), np.int32(2650), np.int32(2985), np.int32(3786), np.int32(4407), np.int32(5157), np.int32(6053), np.int32(7030), np.int32(7369), np.int32(7890)]
actual CPs: [1161, 2070, 3547, 4627, 5403, 6514, 7405, 8737, 9638]
logSize 9409
[0.25, 0.25, 0.5, 0.5, 0.5833333333333334, 0.625, 0.6, 0.5, 0.8571428571428571, 0.6, 0.25, 0.75, 0.3333333333333333, 0.5, 0.6428571428571429, 0.2857142857142857, 0.7142857142857143, 0.6923076923076923, 0.3333333333333333, 0.3333333333333333, 0.6363636363636364, 0.14285714285714285, 0.0, 0.4444444444444444, 1.0, 0.3333333333333333, 0.2, 0.7777777777777778, 0.6111111111111112, 0.0, 0.6, 0.2857142857142857, 0.0, 0.42857142857142855, 0.14285714285714285, 0.7142857142857143, 0.3333333333333333, 0.5, 0.14285714285714285, 0.0, 0.0, 0.16666666666666666, 0.6923076923076923, 0.5, 0.5, 0.42857142857142855, 0.5833333333333334, 0.7272727272727273, 0.375, 0.3333333333333333, 0.14285714285714285, 0.5, 0.2857142857142857, 0.5, 0.4545454

parsing log, completed traces :: 100%|██████████| 8290/8290 [00:01<00:00, 5192.80it/s]


detected CPs: [np.int32(518), np.int32(906), np.int32(1509), np.int32(1884), np.int32(2354), np.int32(3182), np.int32(3927), np.int32(5189), np.int32(5740)]
actual CPs: [1091, 2218, 3016, 4029, 4956, 6397, 7026]
logSize 6974
[0.25, 0.25, 0.5, 0.5, 0.5833333333333334, 0.625, 0.6, 0.5, 0.8571428571428571, 0.6, 0.25, 0.75, 0.3333333333333333, 0.5, 0.6428571428571429, 0.2857142857142857, 0.7142857142857143, 0.6923076923076923, 0.3333333333333333, 0.3333333333333333, 0.6363636363636364, 0.14285714285714285, 0.0, 0.4444444444444444, 1.0, 0.3333333333333333, 0.2, 0.7777777777777778, 0.6111111111111112, 0.0, 0.6, 0.2857142857142857, 0.0, 0.42857142857142855, 0.14285714285714285, 0.7142857142857143, 0.3333333333333333, 0.5, 0.14285714285714285, 0.0, 0.0, 0.16666666666666666, 0.6923076923076923, 0.5, 0.5, 0.42857142857142855, 0.5833333333333334, 0.7272727272727273, 0.375, 0.3333333333333333, 0.14285714285714285, 0.5, 0.2857142857142857, 0.5, 0.45454545454545453, 0.2, 0.4666666666666667, 0.333333

In [39]:
#According to Kraus lag is calculated based on the number of cases in an event log (log_size)
#According to Adamds lag_acc = 200 for all event logs
%run ConceptDriftBPM.ipynb

#testLogs = log_names[0:2]

def apply_kernelCPD(event_log):
    pen = len(event_log)**0.5
    multiTS = multiTimeSeries(event_log) 
    algo = rpt.KernelCPD(kernel="rbf").fit(multiTS)
    change_points = algo.predict(pen=pen) # Penalty controls number of changepoints (higher penalty = fewer changes)
    return change_points

evalFunctionGraph(testLogs, systemPaths, apply_kernelCPD, lags)

 File found: data_collection\datasets_evaluation\without_noise\without_noise_part_1\log_10_1692952190.xes


parsing log, completed traces :: 100%|██████████| 7405/7405 [00:03<00:00, 2163.47it/s]


      case:concept:name                                       concept:name
0                     1         [a, b, g, h, p, g, h, p, g, h, p, g, o, l]
1111                  2  [a, b, c, q, f, b, c, q, j, b, c, q, f, b, c, ...
2222                  3                           [a, b, d, m, d, m, d, n]
3333                  4      [a, b, c, q, e, c, q, j, b, d, m, d, m, d, n]
4444                  5  [a, b, c, q, j, b, c, q, f, b, d, m, d, m, d, ...
...                 ...                                                ...
7114               7401   [a, b, d, m, d, m, d, m, d, c, q, e, c, q, j, n]
7115               7402                     [a, b, b, g, o, k, o, k, o, l]
7116               7403                                    [a, b, g, i, l]
7117               7404      [a, b, b, b, d, c, q, e, c, q, e, c, q, f, n]
7118               7405                                    [b, a, g, i, l]

[7405 rows x 2 columns]
detected CPs: [np.int32(869), np.int32(1372), np.int32(2475), np.int32(2788

parsing log, completed traces :: 100%|██████████| 3391/3391 [00:01<00:00, 1996.29it/s]


      case:concept:name                                       concept:name
0                     1                                          [a, h, e]
1111                  2                                          [a, h, e]
2222                  3  [b, d, m, d, m, d, m, d, m, d, m, d, m, d, n, ...
2725                  4                                          [a, h, e]
2836                  5      [b, d, n, j, q, o, j, q, o, j, q, o, j, q, p]
...                 ...                                                ...
2653               3387  [b, f, l, s, b, d, m, d, n, i, r, d, m, d, m, ...
2654               3388                        [a, g, a, g, a, g, a, h, e]
2655               3389                        [a, g, a, g, a, g, a, h, e]
2657               3390   [b, f, s, l, b, d, m, d, n, i, r, d, n, j, q, p]
2658               3391                  [b, d, m, d, n, j, q, o, j, q, p]

[3391 rows x 2 columns]
detected CPs: [np.int32(225), np.int32(1008), np.int32(1797), np.int32(2010

In [55]:
#According to Kraus lag is calculated based on the number of cases in an event log (log_size)
#According to Adamds lag_acc = 200 for all event logs
%run ConceptDriftBPM.ipynb

lags = 5 #[1, 5, 10]

base = Path("data_collection\datasets_evaluation\with_noise_10")
systemPaths = [
    os.path.join(base, "with_noise_10_part_1"),
    os.path.join(base, "with_noise_10_part_2") ]

testLogs = log_names

def apply_kernelCPD(event_log):
    pen = len(event_log)**0.5
    multiTS = multiTimeSeries(event_log) 
    algo = rpt.KernelCPD(kernel="rbf").fit(multiTS)
    change_points = algo.predict(pen=pen) # Penalty controls number of changepoints (higher penalty = fewer changes)
    return change_points

evalFunctionGraph(testLogs, systemPaths, apply_kernelCPD, lags)

 File found: data_collection\datasets_evaluation\with_noise_10\with_noise_10_part_1\log_10_1692952190.xes


parsing log, completed traces :: 100%|██████████| 7405/7405 [00:02<00:00, 3269.12it/s]


detected CPs: [np.int32(722), np.int32(1100), np.int32(2058), np.int32(2341), np.int32(3328), np.int32(3642), np.int32(4264), np.int32(5159), 5896]
actual CPs: [1029, 1562, 2818, 4306, 5516, 6341]
logSize 6295
[0.2222222222222222]
[0.3333333333333333]
[0.26666666666666666]
 File found: data_collection\datasets_evaluation\with_noise_10\with_noise_10_part_1\log_11_1692952195.xes


parsing log, completed traces :: 100%|██████████| 3059/3059 [00:00<00:00, 9328.40it/s]


detected CPs: [np.int32(381), np.int32(832), np.int32(1308), np.int32(1463), 2205]
actual CPs: [1228, 1853]
logSize 2604
[0.2222222222222222, 0.2]
[0.3333333333333333, 0.5]
[0.26666666666666666, 0.28571428571428575]
 File found: data_collection\datasets_evaluation\with_noise_10\with_noise_10_part_1\log_12_1692952196.xes


...

detected CPs: [np.int32(776), np.int32(1940), np.int32(2062), np.int32(2285), 2375]
actual CPs: [1086, 1956]
logSize 2774
[0.2222222222222222, 0.2, 0.5555555555555556, 0.7142857142857143, 0.5384615384615384, 0.5454545454545454, 0.6363636363636364, 0.5555555555555556, 0.75, 0.5, 0.2]
[0.3333333333333333, 0.5, 0.8333333333333334, 0.625, 0.875, 0.75, 0.6363636363636364, 0.8333333333333334, 0.6666666666666666, 0.5, 0.5]
[0.26666666666666666, 0.28571428571428575, 0.6666666666666667, 0.6666666666666666, 0.6666666666666667, 0.631578947368421, 0.6363636363636364, 0.6666666666666667, 0.7058823529411765, 0.5, 0.28571428571428575]
 File found: data_collection\datasets_evaluation\with_noise_10\with_noise_10_part_1\log_20_1692952238.xes


parsing log, completed traces :: 100%|██████████| 15047/15047 [00:02<00:00, 5122.79it/s]


detected CPs: [np.int32(1033), np.int32(1558), np.int32(6478), np.int32(6848), 12489]
actual CPs: [1357, 2105, 3482, 4422, 5484, 6787, 8017, 9381, 10872, 11904, 12802, 13833]
logSize 12888
[0.2222222222222222, 0.2, 0.5555555555555556, 0.7142857142857143, 0.5384615384615384, 0.5454545454545454, 0.6363636363636364, 0.5555555555555556, 0.75, 0.5, 0.2, 0.8]
[0.3333333333333333, 0.5, 0.8333333333333334, 0.625, 0.875, 0.75, 0.6363636363636364, 0.8333333333333334, 0.6666666666666666, 0.5, 0.5, 0.3333333333333333]
[0.26666666666666666, 0.28571428571428575, 0.6666666666666667, 0.6666666666666666, 0.6666666666666667, 0.631578947368421, 0.6363636363636364, 0.6666666666666667, 0.7058823529411765, 0.5, 0.28571428571428575, 0.47058823529411764]
 File found: data_collection\datasets_evaluation\with_noise_10\with_noise_10_part_1\log_21_1692952246.xes


parsing log, completed traces :: 100%|██████████| 6924/6924 [00:00<00:00, 8959.09it/s]


KeyboardInterrupt: 

In [ ]:
#According to Kraus lag is calculated based on the number of cases in an event log (log_size)
#According to Adamds lag_acc = 200 for all event logs
%run ConceptDriftBPM.ipynb

lags = 5 #[1, 5, 10]

base = Path("data_collection\datasets_evaluation\without_noise")
systemPaths = [
    os.path.join(base, "without_noise_part_1"),
    os.path.join(base, "without_noise_part_2") ]

testLogs = log_names

def apply_kernelCPD(event_log):
    pen = len(event_log)**0.5
    multiTS = multiTimeSeries(event_log) 
    algo = rpt.KernelCPD(kernel="rbf").fit(multiTS)
    change_points = algo.predict(pen=pen) # Penalty controls number of changepoints (higher penalty = fewer changes)
    return change_points

evalFunctionGraph(testLogs, systemPaths, apply_kernelCPD, lags)

In [18]:
#Test
#logTest = pm4py.read.read_xes("data_collection\datasets_evaluation\without_noise\without_noise_part_2\log_100_1692952649.xes")
#logXES = logTest.groupby(['case:concept:name'])['concept:name'].apply(list).reset_index()
#logXES["case:concept:name"] = logXES["case:concept:name"].apply(lambda x: int(x))
#logXES = logXES.sort_values(by=["case:concept:name"])
#logXES

#%run ConceptDriftBPM.ipynb
#test = logXES['concept:name'][0:10].reset_index(drop=True)
#test[0]
#df_list_multi(test)

['*a',
 'ah',
 'he',
 'e$',
 '*a',
 'ah',
 'he',
 'e$',
 '*b',
 'bd',
 'dm',
 'md',
 'dm',
 'md',
 'dm',
 'md',
 'dm',
 'md',
 'dm',
 'md',
 'dm',
 'md',
 'dn',
 'ni',
 'ir',
 'rd',
 'dn',
 'nj',
 'jq',
 'qp',
 'p$',
 '*a',
 'ah',
 'he',
 'e$',
 '*b',
 'bd',
 'dn',
 'nj',
 'jq',
 'qo',
 'oj',
 'jq',
 'qo',
 'oj',
 'jq',
 'qo',
 'oj',
 'jq',
 'qp',
 'p$',
 '*a',
 'ag',
 'ga',
 'ah',
 'he',
 'e$',
 '*a',
 'ag',
 'ga',
 'ag',
 'ga',
 'ah',
 'he',
 'e$',
 '*a',
 'ah',
 'he',
 'e$',
 '*a',
 'ah',
 'he',
 'e$',
 '*a',
 'ag',
 'ga',
 'ah',
 'he',
 'e$']

In [109]:
#1 approach
%run Algo2_Bose.ipynb

def apply_bose_j(event_log):
    windowSize = 150
    step_size = 2
    pvals_j = detectChange_JMeasure_KS_Step(log=event_log, windowSize=windowSize, step_size=step_size)
    cp_j = visualInspection_Step(pvals_j, windowSize)
    return cp_j

#According to Kraus lag is calculated based on the number of cases in an event log (log_size)
#According to Adamds lag_acc = 200 for all event logs
lags = 5 #[1, 5, 10]

base = Path("data_collection\datasets_evaluation\without_noise")
systemPaths = [
    os.path.join(base, "without_noise_part_1"),
    os.path.join(base, "without_noise_part_2") ]


log_names = goldStandard.log_name.tolist()
testLogs = log_names[0:10]




evalFunction(testLogs, systemPaths, apply_bose_j, lags)


#Test
#lag = 5
#log_size = len(logXES)
#lag_acc = int(log_size / 100 * lag)
#precision, recall = calcPrecisionRecall(a, actualCP, lag_acc)

 File found: data_collection\datasets_evaluation\without_noise\without_noise_part_1\log_10_1692952190.xes


parsing log, completed traces :: 100%|██████████| 7405/7405 [00:03<00:00, 2290.62it/s]
Calculating J P-Values for Bose, activity pairs complete :: 100%|██████████| 324/324 [11:41<00:00,  2.17s/it]


detected CPs: [np.int64(1505), np.int64(2215), np.int64(2840)]
actual CPs: [1029, 1562, 2818, 4306, 5516, 6341]
logSize 7405
[0.6666666666666666]
[0.3333333333333333]
[0.4444444444444444]
 File found: data_collection\datasets_evaluation\without_noise\without_noise_part_1\log_11_1692952195.xes


parsing log, completed traces :: 100%|██████████| 3059/3059 [00:00<00:00, 7164.13it/s]
Calculating J P-Values for Bose, activity pairs complete :: 100%|██████████| 36/36 [00:28<00:00,  1.26it/s]


detected CPs: []
actual CPs: [1228, 1853]
logSize 3059
[0.6666666666666666, 0]
[0.3333333333333333, 0.0]
[0.4444444444444444, 0]
 File found: data_collection\datasets_evaluation\without_noise\without_noise_part_1\log_12_1692952196.xes


...

detected CPs: [np.int64(812), np.int64(2488), np.int64(3108)]
actual CPs: [1444, 2772, 3489, 4826, 6062, 6751]
logSize 8012
[0.6666666666666666, 0, 0, 0.5, 0.5, 1.0, 0.8, 0.6666666666666666]
[0.3333333333333333, 0.0, 0.0, 0.25, 0.25, 0.25, 0.36363636363636365, 0.3333333333333333]
[0.4444444444444444, 0, 0, 0.3333333333333333, 0.3333333333333333, 0.4, 0.5000000000000001, 0.4444444444444444]
 File found: data_collection\datasets_evaluation\without_noise\without_noise_part_1\log_18_1692952223.xes


parsing log, completed traces :: 100%|██████████| 11493/11493 [00:02<00:00, 4852.90it/s]
Calculating J P-Values for Bose, activity pairs complete :: 100%|██████████| 196/196 [11:29<00:00,  3.52s/it]


detected CPs: [np.int64(3588), np.int64(4464)]
actual CPs: [1389, 2280, 3635, 4421, 5728, 7035, 7597, 8723, 9996]
logSize 11493
[0.6666666666666666, 0, 0, 0.5, 0.5, 1.0, 0.8, 0.6666666666666666, 1.0]
[0.3333333333333333, 0.0, 0.0, 0.25, 0.25, 0.25, 0.36363636363636365, 0.3333333333333333, 0.2222222222222222]
[0.4444444444444444, 0, 0, 0.3333333333333333, 0.3333333333333333, 0.4, 0.5000000000000001, 0.4444444444444444, 0.3636363636363636]
 File found: data_collection\datasets_evaluation\without_noise\without_noise_part_2\log_100_1692952649.xes


parsing log, completed traces :: 100%|██████████| 3391/3391 [00:01<00:00, 2404.24it/s]
Calculating J P-Values for Bose, activity pairs complete :: 100%|██████████| 361/361 [06:48<00:00,  1.13s/it]

detected CPs: []
actual CPs: [1399, 2176]
logSize 3391
[0.6666666666666666, 0, 0, 0.5, 0.5, 1.0, 0.8, 0.6666666666666666, 1.0, 0]
[0.3333333333333333, 0.0, 0.0, 0.25, 0.25, 0.25, 0.36363636363636365, 0.3333333333333333, 0.2222222222222222, 0.0]
[0.4444444444444444, 0, 0, 0.3333333333333333, 0.3333333333333333, 0.4, 0.5000000000000001, 0.4444444444444444, 0.3636363636363636, 0]
------------------
Evaluation Measures: <function apply_bose_j at 0x00000138FF72F100>
------------------
precision: 0.5133333333333333 -> full list: [0.6666666666666666, 0, 0, 0.5, 0.5, 1.0, 0.8, 0.6666666666666666, 1.0, 0]
recall: 0.20025252525252527 -> full list: [0.3333333333333333, 0.0, 0.0, 0.25, 0.25, 0.25, 0.36363636363636365, 0.3333333333333333, 0.2222222222222222, 0.0]
f1: 0.28191919191919196 -> full list [0.4444444444444444, 0, 0, 0.3333333333333333, 0.3333333333333333, 0.4, 0.5000000000000001, 0.4444444444444444, 0.3636363636363636, 0]
------------------


In [91]:
# TEST

a = [np.int64(801),
 np.int64(892),
 np.int64(982),
 np.int64(1745),
 np.int64(2338),
 np.int64(2518),
 np.int64(3162),
 np.int64(3856),
 np.int64(4749)]

actualCP = goldStandard.loc[goldStandard['log_name'] == 'log_6_1692952173.xes','change_point'].explode().tolist()
#actualCP

lag = 5
log_size = len(logXES)
lag_acc = int(log_size / 100 * lag)
precision, recall = calcPrecisionRecall(a, actualCP, lag_acc)

precision, recall

[1350, 1920, 3342, 4346, 5008, 6173, 7280, 8196, 9350]